# FlavourNet: from coffee shop reviews to a knowledge graph

This notebook turns 10,000 Yelp reviews of Coffee & Tea businesses into a network of concepts (drinks, food, service, ambience, quality words) and the relationships customers draw between them, then analyses that network.

Pipeline: **preprocess → extract concepts (TF-IDF, YAKE, spaCy NER) → extract relationships (dependency parsing) → filter by PMI → build the graph → centrality, communities and sentiment**.

Input: `data/coffee_tea_sample.csv`, built from the Yelp Open Dataset by `python src/build_sample.py` (the reviews themselves are not redistributed; see the README).

In [ ]:
import math
import re
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import community as community_louvain  # python-louvain
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import spacy
import yake
from sklearn.feature_extraction.text import TfidfVectorizer
from textblob import TextBlob
from wordcloud import WordCloud

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUTPUTS = ROOT / "data", ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

nlp = spacy.load("en_core_web_sm")
SEED = 42

## 1. Load and preprocess the reviews

Lower-case, strip everything that is not a letter, then lemmatise with spaCy and drop stop words and tokens of two letters or fewer. `clean_text` feeds TF-IDF; the raw text is kept for YAKE, NER and dependency parsing, which need real sentences.

In [ ]:
reviews = pd.read_csv(DATA / "coffee_tea_sample.csv")
print(f"{len(reviews):,} reviews from {reviews.business_name.nunique():,} businesses in "
      f"{reviews.state.nunique()} states, {reviews.date.min()[:4]}-{reviews.date.max()[:4]}")
reviews.stars.value_counts().sort_index()

In [ ]:
def preprocess(doc):
    return [t.lemma_ for t in doc
            if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha and len(t.text) > 2]

cleaned = (re.sub(r"[^a-zA-Z\s]", " ", str(t).lower()) for t in reviews.text)
reviews["tokens"] = [preprocess(doc) for doc in nlp.pipe(cleaned, batch_size=200)]
reviews["clean_text"] = reviews.tokens.str.join(" ")
print("Example tokens:", reviews.tokens[0][:10])

## 2. Concept extraction

Three complementary methods, combined into one vocabulary of concepts:

- **TF-IDF** (top 300 unigrams and bigrams) finds terms that characterise individual reviews
- **YAKE** finds key phrases in each raw review without needing a corpus; the 100 most frequent are kept
- **spaCy NER** picks up named things: brands, places and products (top 50)

In [ ]:
tfidf = TfidfVectorizer(max_features=300, ngram_range=(1, 2), min_df=5, max_df=0.85)
tfidf_matrix = tfidf.fit_transform(reviews.clean_text).tocsr()
terms = tfidf.get_feature_names_out()
tfidf_keywords = set(terms)

tfidf_per_review = pd.DataFrame(
    [{"review_id": reviews.review_id.iat[i], "concept": terms[j], "tfidf_score": round(s, 6)}
     for i in range(tfidf_matrix.shape[0])
     for j, s in zip(tfidf_matrix[i].indices, tfidf_matrix[i].data)])
tfidf_global = (tfidf_per_review.groupby("concept").tfidf_score.mean()
                .sort_values(ascending=False).rename("mean_tfidf").reset_index())
concept_importance = dict(zip(tfidf_global.concept, tfidf_global.mean_tfidf))

print(f"TF-IDF concepts: {len(tfidf_keywords)}")
print("Highest mean TF-IDF:", tfidf_global.concept.head(15).tolist())

In [ ]:
kw_extractor = yake.KeywordExtractor(lan="en", n=2, dedupLim=0.7, top=15)
yake_df = pd.DataFrame(
    [{"review_id": rid, "concept": kw.lower(), "yake_importance": round(1 - score, 4)}
     for rid, text in zip(reviews.review_id, reviews.text)
     for kw, score in kw_extractor.extract_keywords(str(text))])
yake_freq = Counter(yake_df.concept)
top_yake = {kw for kw, _ in yake_freq.most_common(100)}
print(f"YAKE concepts: {len(top_yake)}")
pd.Series(dict(yake_freq.most_common(15)), name="reviews").to_frame().T

In [ ]:
NER_LABELS = {"ORG", "PRODUCT", "GPE", "FAC", "NORP"}
reviews["ner_entities"] = [[(e.text.lower(), e.label_) for e in doc.ents if e.label_ in NER_LABELS]
                           for doc in nlp.pipe(reviews.text.astype(str), batch_size=200)]
ner_freq = Counter(e for ents in reviews.ner_entities for e, _ in ents)
top_ner = {e for e, _ in ner_freq.most_common(50)}

all_concepts = tfidf_keywords | top_yake | top_ner
token_freq = Counter(t for tokens in reviews.tokens for t in tokens)
print(f"NER entities: {len(top_ner)}  ->  combined vocabulary: {len(all_concepts)} concepts")

Each concept gets a type from a hand-built coffee-shop lexicon. Anything not in the lexicon is a general `CONCEPT`.

In [ ]:
ENTITY_CATEGORIES = {
    "DRINK": {"coffee", "espresso", "latte", "cappuccino", "matcha", "tea", "americano", "mocha", "chai",
              "cortado", "flat white", "macchiato", "frappuccino", "cold brew", "iced coffee", "boba", "taro",
              "hot chocolate", "juice", "smoothie"},
    "FOOD": {"pastry", "croissant", "bagel", "sandwich", "cake", "muffin", "toast", "cookie", "waffle", "scone",
             "pancake", "egg", "donut", "biscuit", "quiche", "granola", "oatmeal", "brownie", "salad", "wrap",
             "avocado", "cream", "cheese", "bread", "chicken"},
    "SERVICE": {"barista", "staff", "service", "server", "cashier", "friendly", "attentive", "helpful",
                "efficient", "rude", "slow", "fast", "wait", "queue", "order"},
    "AMBIENCE": {"atmosphere", "ambience", "cozy", "noisy", "quiet", "seating", "decor", "music", "lighting",
                 "outdoor", "indoor", "wifi", "space", "parking", "clean", "busy", "crowded"},
    "EXPERIENCE": {"work", "study", "morning", "weekend", "visit", "brunch", "lunch", "breakfast", "relax",
                   "hangout", "date", "meeting"},
    "QUALITY": {"good", "great", "delicious", "amazing", "fresh", "hot", "cold", "sweet", "excellent",
                "awesome", "perfect", "strong", "weak", "expensive", "cheap", "bad", "terrible", "poor", "warm",
                "bitter", "tasty", "wonderful", "crispy", "fluffy", "creamy", "rich", "smooth", "bold", "mild",
                "overpriced", "affordable", "outstanding", "mediocre", "decent", "disappointing"},
}
CONCEPT_TYPE = {w: cat for cat, words in ENTITY_CATEGORIES.items() for w in words}

def get_type(concept):
    return CONCEPT_TYPE.get(concept.lower(), "CONCEPT")

concepts_df = (pd.DataFrame([{"concept": c, "frequency": token_freq.get(c, 0), "entity_type": get_type(c)}
                             for c in all_concepts])
               .sort_values("frequency", ascending=False, ignore_index=True))
concepts_df.entity_type.value_counts()

## 3. Relationship extraction

For every sentence, each token whose grammatical link to its head is one of a set of useful dependency types (adjective modifier, subject, object, compound, preposition, attribute, conjunction) becomes a candidate edge **token → head**, kept only if both ends are in the concept vocabulary. The dependency type names the relation, so *"friendly staff"* gives `friendly --described_as--> staff`.

In [ ]:
VALID_DEPS = {"amod", "nsubj", "nsubjpass", "dobj", "compound", "prep", "pobj", "attr", "conj"}
SKIP = {"the", "and", "but", "for", "not", "are", "was"}

def classify_relation(subj, obj, dep):
    if dep == "amod": return "described_as"
    if dep == "nsubj": return "has_property"
    if dep == "dobj": return "relates_to"
    if dep == "compound": return "associated_with"
    if dep in ("prep", "pobj"): return "connected_to"
    st, ot = get_type(subj), get_type(obj)
    if st == "QUALITY" and ot in ("DRINK", "FOOD", "SERVICE"): return "describes"
    if ot == "QUALITY" and st in ("DRINK", "FOOD", "SERVICE"): return "has_quality"
    if st in ("DRINK", "FOOD") and ot in ("DRINK", "FOOD"): return "associated_with"
    if st in ("DRINK", "FOOD") and ot == "SERVICE": return "served_by"
    if st in ("SERVICE", "AMBIENCE") and ot == "EXPERIENCE": return "affects"
    return "relates_to"

dep_edges = defaultdict(lambda: {"count": 0, "reviews": set(), "relations": Counter()})
for rid, doc in zip(reviews.review_id, nlp.pipe(reviews.text.astype(str), batch_size=200)):
    for token in doc:
        if token.dep_ not in VALID_DEPS:
            continue
        subj, obj = token.lemma_.lower().strip(), token.head.lemma_.lower().strip()
        if (subj not in all_concepts or obj not in all_concepts or subj == obj
                or len(subj) < 3 or len(obj) < 3 or subj in SKIP):
            continue
        e = dep_edges[subj, obj]
        e["count"] += 1
        e["reviews"].add(rid)
        e["relations"][classify_relation(subj, obj, token.dep_)] += 1
print(f"Candidate concept pairs: {len(dep_edges):,}")

### Filtering with PMI

Most candidate pairs are noise or simply two very common words. **Pointwise mutual information** compares how often two concepts appear in the same review with how often they would by chance: PMI > 0 means they co-occur more than expected. An edge is kept if PMI > 0 and it is seen at least twice, in at least two different reviews.

In [ ]:
dep_df = pd.DataFrame([{
    "source": s, "target": t,
    "relation": d["relations"].most_common(1)[0][0],
    "extraction_method": "dependency_parsing",
    "source_type": get_type(s), "target_type": get_type(t),
    "weight": d["count"], "review_count": len(d["reviews"]),
} for (s, t), d in dep_edges.items()])

concept_reviews = tfidf_per_review.groupby("concept").review_id.nunique().to_dict()
n_reviews = tfidf_per_review.review_id.nunique()

def pmi(row):
    p_xy = row.review_count / n_reviews
    p_x = concept_reviews.get(row.source, 1) / n_reviews
    p_y = concept_reviews.get(row.target, 1) / n_reviews
    return round(math.log2(p_xy / (p_x * p_y)), 4)

dep_df["pmi_score"] = dep_df.apply(pmi, axis=1)
dep_df["source_importance"] = dep_df.source.map(concept_importance).fillna(0).round(6)
dep_df["target_importance"] = dep_df.target.map(concept_importance).fillna(0).round(6)

edges = dep_df[(dep_df.pmi_score > 0) & (dep_df.weight >= 2) & (dep_df.review_count >= 2)].reset_index(drop=True)
w = edges.weight
edges["normalized_weight"] = ((w - w.min()) / (w.max() - w.min())).round(6)

print(f"{len(dep_df):,} candidate edges -> {len(edges)} after the PMI filter")
print(edges.relation.value_counts().to_string())
edges.sort_values("weight", ascending=False)[["source", "relation", "target", "weight", "pmi_score"]].head(12)

## 4. The knowledge graph

A directed graph: nodes are concepts (with type, frequency and TF-IDF importance), edges are the filtered relationships.

In [ ]:
G = nx.DiGraph()
for node in set(edges.source) | set(edges.target):
    G.add_node(node, entity_type=get_type(node), frequency=token_freq.get(node, 0),
               importance=round(concept_importance.get(node, 0), 6))
for r in edges.itertuples():
    G.add_edge(r.source, r.target, relation=r.relation, weight=r.weight, pmi_score=r.pmi_score,
               normalized_weight=r.normalized_weight, review_count=r.review_count,
               source_type=r.source_type, target_type=r.target_type)

print(f"Nodes {G.number_of_nodes()}, edges {G.number_of_edges()}, density {nx.density(G):.4f}")
pd.Series(Counter(nx.get_node_attributes(G, "entity_type").values())).sort_values(ascending=False)